In [ ]:
# Gemini
import re
from typing import List

class CGPOFormat():
    def __init__(self):
        self.task_setags = {
            'seg': ('<contour_list>', '</contour_list>'),
            'det_no_class': ('<bbox_list>', '</bbox_list>'),
            'det_with_class': ('<detection_result>', '</detection_result>')
        }
        self.stag_set = set(token for token, _ in self.task_setags.values())
        # 你的原始正则
        self.basic_pattern = r'^<think>.*?</think>.*?<answer>.*?</answer>(?![\s\S])'

    def __call__(self, completions, solution, task, messages, **kwargs) -> List[float]:
        rewards = []
        for content, gt, task_type, msgs in zip(completions, solution, task, messages):
            format_reward = None

            if task_type == 'choice_rsn_loc':
                # --- 补全的逻辑开始 ---
                # 1. 基础格式检查 (0.5分)
                mch = re.match(self.basic_pattern, content, re.DOTALL | re.MULTILINE)
                format_reward = 0.5 if mch else 0.0

                if format_reward > 0:
                    # 提取所有 entity 标签及其信息
                    entity_iter = re.finditer(r'<entity name="([^"]+)" id="([^"]+)">\s*(.*?)\s*</entity>', content, re.DOTALL)
                    entities = [{
                        'name': m.group(1),
                        'id': m.group(2),
                        'content': m.group(3),
                        'start': m.start(),
                        'end': m.end()
                    } for m in entity_iter]

                    # 2. 检查 entity 存在且 ID 从1开始递增 (0.6分)
                    ids_valid = False
                    if entities:
                        try:
                            ids = [int(e['id']) for e in entities]
                            if ids == list(range(1, len(entities) + 1)):
                                ids_valid = True
                        except ValueError:
                            pass 
                    
                    if ids_valid:
                        format_reward = 0.6
                        
                        # 3. 检查每个 entity 内部是否有有效 bbox (0.7分)
                        bbox_valid = all(
                            re.search(r'<bbox_list>\s*(?:<box>.*?</box>\s*)+</bbox_list>', e['content'], re.DOTALL)
                            for e in entities
                        )
                        
                        if bbox_valid:
                            format_reward = 0.7
                            
                            # 4. 检查所有 entity 是否在 <think> 内部 (0.8分)
                            think_end_idx = content.find('</think>')
                            if entities[-1]['end'] < think_end_idx:
                                format_reward = 0.8
                                
                                # 5. 检查引用完整性 (1.0分)
                                ref_iter = re.finditer(r'<ref name="([^"]+)" id="([^"]+)"/>', content)
                                refs = [{'name': m.group(1), 'id': m.group(2), 'start': m.start()} for m in ref_iter]
                                
                                all_refs_valid = True
                                for e in entities:
                                    # 找到指向当前 entity ID 的所有 ref
                                    my_refs = [r for r in refs if r['id'] == e['id']]
                                    
                                    # a: 必须至少有一个 ref
                                    if not my_refs:
                                        all_refs_valid = False
                                        break
                                    
                                    # b: ref 必须在 entity 后面且 name 一致
                                    for r in my_refs:
                                        if r['name'] != e['name'] or r['start'] < e['end']:
                                            all_refs_valid = False
                                            break
                                    
                                    if not all_refs_valid:
                                        break
                                
                                if all_refs_valid:
                                    format_reward = 1.0
                # --- 补全的逻辑结束 ---

            else:
                format_reward = 0.0 # 简化其他分支用于测试

            rewards.append(format_reward)
        return rewards

In [1]:
# GPT
import re
from typing import List

def choice_rsn_loc_format_reward(content, basic_pattern):
    format_reward = None
    # 1) 基础格式：<think>…</think>…<answer>…</answer>
    mch = re.match(basic_pattern, content, re.DOTALL | re.MULTILINE)
    if not mch:
        format_reward = 0.0
    else:
        format_reward = 0.5

        # 提取think范围（用于条件4）
        t_m = re.search(r'<think>(?P<t>.*?)</think>', content, re.DOTALL)
        think_span = (t_m.start(), t_m.end()) if t_m else None

        # 抽取所有entity
        ent_pat = re.compile(
            r'<entity\s+name="(?P<name>[^"]+)"\s+id="(?P<id>\d+)">\s*(?P<body>.*?)\s*</entity>',
            re.DOTALL
        )
        entities = list(ent_pat.finditer(content))
        if not entities:
            format_reward = 0.5
        else:
            ids = [int(m.group('id')) for m in entities]

            # 2) id必须从1开始严格递增
            if ids == list(range(1, len(ids) + 1)):
                format_reward = 0.6

                # 3) 每个entity内“只有bbox_list”（可有空白），且bbox_list里至少一个box；
                #    同时校验每个box为x1,y1,x2,y2整数且0-1000，并满足x1<x2,y1<y2
                def _valid_box_text(s: str) -> bool:
                    m = re.fullmatch(r'\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*', s)
                    if not m:
                        return False
                    x1, y1, x2, y2 = map(int, m.groups())
                    if not all(0 <= v <= 1000 for v in (x1, y1, x2, y2)):
                        return False
                    return (x1 < x2) and (y1 < y2)

                ok_bbox = True
                for m in entities:
                    body = m.group('body')

                    # 3.a 去掉<bbox_list>…</bbox_list>后，剩余必须全是空白（否则视为脏数据，止步0.6）
                    body_wo_bbox = re.sub(r'<bbox_list>.*?</bbox_list>', '', body, flags=re.DOTALL).strip()
                    if body_wo_bbox:
                        ok_bbox = False
                        break

                    # 3.b 必须存在<bbox_list>且至少一个<box>
                    b = re.search(r'<bbox_list>(?P<b>.*?)</bbox_list>', body, re.DOTALL)
                    if not b:
                        ok_bbox = False
                        break
                    box_texts = re.findall(r'<box>\s*([^<]+?)\s*</box>', b.group('b'))
                    if not box_texts:
                        ok_bbox = False
                        break

                    # 3.c 校验每个box坐标合法性；任一不合法则整体不满足条件3（止步0.6）
                    if any(not _valid_box_text(t) for t in box_texts):
                        ok_bbox = False
                        break

                if ok_bbox:
                    format_reward = 0.7

                    # 4) 所有entity必须都在<think>…</think>内
                    ok_in_think = bool(think_span)
                    if ok_in_think:
                        ts, te = think_span
                        ok_in_think = all(ts <= m.start() and m.end() <= te for m in entities)
                    if ok_in_think:
                        format_reward = 0.8

                        # 5) ref规则：每个entity后至少一个ref；ref不得在对应entity之前；name一致；
                        #    且任何ref的id若不属于entity集合，则不能到1.0（保持0.8）
                        ref_pat = re.compile(r'<ref\s+name="(?P<name>[^"]+)"\s+id="(?P<id>\d+)"\s*/>')
                        refs = list(ref_pat.finditer(content))

                        ent_id_set = set(ids)
                        ref_id_set = {int(r.group('id')) for r in refs}
                        has_unknown_ref = len(ref_id_set - ent_id_set) > 0

                        refs_by_id = {}
                        for r in refs:
                            rid = int(r.group('id'))
                            refs_by_id.setdefault(rid, []).append((r.start(), r.group('name')))

                        ok_ref = True
                        for ent in entities:
                            eid = int(ent.group('id'))
                            ename = ent.group('name')
                            estart, eend = ent.start(), ent.end()

                            # ref不能出现在对应entity之前
                            if any(pos < estart for pos, _ in refs_by_id.get(eid, [])):
                                ok_ref = False
                                break
                            # entity之后至少一个ref
                            after_refs = [(pos, rname) for pos, rname in refs_by_id.get(eid, []) if pos > eend]
                            if not after_refs:
                                ok_ref = False
                                break
                            # 所有ref的name必须与entity的name完全一致
                            if any(rname != ename for _, rname in after_refs):
                                ok_ref = False
                                break

                        if ok_ref and (not has_unknown_ref):
                            format_reward = 1.0
    return format_reward

class CGPOFormat():

    def __init__(self):
        self.task_setags = {
            'seg': ('<contour_list>', '</contour_list>'),
            'det_no_class': ('<bbox_list>', '</bbox_list>'),
            'det_with_class': ('<detection_result>', '</detection_result>')
        }
        self.stag_set = set(token for token, _ in self.task_setags.values())
        self.basic_pattern = r'^<think>.*?</think>.*?<answer>.*?</answer>(?![\s\S])'

    def __call__(self, completions, solution, task, messages, **kwargs) -> List[float]:
        rewards = []
        for content, gt, task_type, msgs in zip(completions, solution, task, messages):
            format_reward = None

            if task_type in ['choice', 'choice_func']:
                mch = re.match(self.basic_pattern, content, re.DOTALL | re.MULTILINE)
                format_reward = 1.0 if mch else 0.0
                if task_type == 'choice_func' and not any(m['role'] == 'tool' for m in msgs) and format_reward > 0.5:
                    format_reward = 0.5
            elif task_type in ['choice_nothink', 'choice_nothink_func']:
                if '<think>' in content or '</think>' in content or '<answer>' in content or '</answer>' in content:
                    format_reward = 0.0
                else:
                    format_reward = 1.0
                if task_type == 'choice_nothink_func' and not any(m['role'] == 'tool' for m in msgs) and format_reward > 0.5:
                    format_reward = 0.5
            elif task_type in ['choice_rsn_loc']:
                format_reward = choice_rsn_loc_format_reward(content, self.basic_pattern)
            elif task_type in ['seg', 'det_no_class', 'det_with_class']:
                stag, etag = self.task_setags.get(task_type, (None, None))
                format_reward = 0.1
                if content.count(stag) == 1 and content.count(etag) == 1 \
                    and content.index(stag) < content.index(etag):
                    format_reward = 1.0
                elif check_negative_exist(content):
                    format_reward = 1.0
                elif check_other_task_tag_exist(content, self.stag_set, stag):
                    format_reward = 0.0
            else:
                raise ValueError(f'Not implement format reward for task "{task_type}"')
            
            assert format_reward is not None
            rewards.append(format_reward)
        
        return rewards

In [2]:
# --- Gemini基础测试用例定义 ---

def run_tests():
    scorer = CGPOFormat()
    
    # 通用参数
    task = ['choice_rsn_loc']
    solution = ['D']
    messages = [[{'role': 'user', 'content': 'test'}]]

    test_cases = [
        {
            "desc": "0.0分 - 基础格式错误 (缺少think/answer)",
            "content": "直接回答 D",
            "expected": 0.0
        },
        {
            "desc": "0.5分 - 基础格式正确，但无entity",
            "content": "<think>思考过程...</think>结论<answer>D</answer>",
            "expected": 0.5
        },
        {
            "desc": "0.5分 - ID错误 (ID不是从1开始或不连续)",
            "content": """<think>
            <entity name="细胞" id="2"><bbox_list><box>0,0,10,10</box></bbox_list></entity>
            </think><answer>D</answer>""",
            "expected": 0.5
        },
        {
            "desc": "0.6分 - ID正确，但缺少bbox或bbox格式错误",
            "content": """<think>
            <entity name="细胞" id="1">这里没有box list</entity>
            </think><answer>D</answer>""",
            "expected": 0.6
        },
        {
            "desc": "0.7分 - bbox正确，但entity在think外部",
            "content": """<think>思考...</think>
            外部定义: <entity name="细胞" id="1"><bbox_list><box>0,0,1,1</box></bbox_list></entity>
            <answer>D</answer>""",
            "expected": 0.7
        },
        {
            "desc": "0.8分 - entity在think内，但缺少引用(ref)",
            "content": """<think>
            发现<entity name="细胞" id="1"><bbox_list><box>0,0,1,1</box></bbox_list></entity>
            但是没有引用它。
            </think><answer>D</answer>""",
            "expected": 0.8
        },
        {
            "desc": "0.8分 - 有引用，但引用ID匹配但Name不匹配",
            "content": """<think>
            <entity name="细胞" id="1"><bbox_list><box>0,0,1,1</box></bbox_list></entity>
            错误引用: <ref name="错误的细胞名" id="1"/>
            </think><answer>D</answer>""",
            "expected": 0.8
        },
        {
            "desc": "0.8分 - 有引用，但引用出现在定义之前 (顺序错误)",
            "content": """<think>
            提前引用: <ref name="细胞" id="1"/>
            <entity name="细胞" id="1"><bbox_list><box>0,0,1,1</box></bbox_list></entity>
            </think><answer>D</answer>""",
            "expected": 0.8
        },
        {
            "desc": "1.0分 - 完美符合所有要求 (单实体)",
            "content": """<think>
            发现<entity name="细胞" id="1"><bbox_list><box>10,10,20,20</box></bbox_list></entity>
            这就是<ref name="细胞" id="1"/>。
            </think><answer>D</answer>""",
            "expected": 1.0
        },
        {
            "desc": "1.0分 - 完美符合所有要求 (多实体+多次引用)",
            "content": """<think>
            首先看到<entity name="A" id="1"><bbox_list><box>1,1,2,2</box></bbox_list></entity>。
            其次看到<entity name="B" id="2"><bbox_list><box>3,3,4,4</box></bbox_list></entity>。
            对比<ref name="A" id="1"/>和<ref name="B" id="2"/>，
            再次观察<ref name="A" id="1"/>。
            </think><answer>D</answer>""",
            "expected": 1.0
        }
    ]

    print(f"{'测试描述':<40} | {'预期':<5} | {'实际':<5} | {'结果'}")
    print("-" * 65)
    
    all_passed = True
    for case in test_cases:
        completions = [case["content"]]
        # 调用奖励函数
        scores = scorer(completions, solution, task, messages)
        actual = scores[0]
        
        status = "PASS" if abs(actual - case["expected"]) < 1e-6 else "FAIL"
        if status == "FAIL": all_passed = False
        
        print(f"{case['desc']:<40} | {case['expected']:<5} | {actual:<5} | {status}")

    print("-" * 65)
    if all_passed:
        print("\n✅ 所有测试用例均通过！奖励逻辑符合预期。")
    else:
        print("\n❌ 存在失败的测试用例，请检查逻辑。")

run_tests()

测试描述                                     | 预期    | 实际    | 结果
-----------------------------------------------------------------
0.0分 - 基础格式错误 (缺少think/answer)           | 0.0   | 0.0   | PASS
0.5分 - 基础格式正确，但无entity                   | 0.5   | 0.5   | PASS
0.5分 - ID错误 (ID不是从1开始或不连续)               | 0.5   | 0.5   | PASS
0.6分 - ID正确，但缺少bbox或bbox格式错误             | 0.6   | 0.6   | PASS
0.7分 - bbox正确，但entity在think外部            | 0.7   | 0.7   | PASS
0.8分 - entity在think内，但缺少引用(ref)          | 0.8   | 0.8   | PASS
0.8分 - 有引用，但引用ID匹配但Name不匹配               | 0.8   | 0.8   | PASS
0.8分 - 有引用，但引用出现在定义之前 (顺序错误)             | 0.8   | 0.8   | PASS
1.0分 - 完美符合所有要求 (单实体)                    | 1.0   | 1.0   | PASS
1.0分 - 完美符合所有要求 (多实体+多次引用)               | 1.0   | 1.0   | PASS
-----------------------------------------------------------------

✅ 所有测试用例均通过！奖励逻辑符合预期。


In [3]:
# --- Gemini复杂的测试用例集 ---

def run_tests():
    scorer = CGPOFormat()
    
    # 基础配置
    task = ['choice_rsn_loc']
    solution = ['D']
    messages = [[{'role': 'user', 'content': 'test'}]]

    # 颜色代码，用于输出美化
    GREEN = "\033[92m"
    RED = "\033[91m"
    RESET = "\033[0m"

    complex_cases = [
        # --- ID 逻辑陷阱 ---
        {
            "desc": "陷阱: ID重复 (1, 1, 2)",
            "content": """<think>
            <entity name="A" id="1"><bbox_list><box>x</box></bbox_list></entity>
            <entity name="B" id="1"><bbox_list><box>x</box></bbox_list></entity>
            <entity name="C" id="2"><bbox_list><box>x</box></bbox_list></entity>
            </think><answer>D</answer>""",
            "expected": 0.5 # ID不满足 range(1, 4)
        },
        {
            "desc": "陷阱: ID跳跃 (1, 3)",
            "content": """<think>
            <entity name="A" id="1"><bbox_list><box>x</box></bbox_list></entity>
            <entity name="B" id="3"><bbox_list><box>x</box></bbox_list></entity>
            </think><answer>D</answer>""",
            "expected": 0.5
        },
        
        # --- BBox 鲁棒性测试 ---
        {
            "desc": "鲁棒性: Entity内有换行和杂乱文本 (应判定<bbox_list>失败)",
            "content": """<think>
            <entity name="脏数据" id="1">
                这是一些描述文本
                <bbox_list>
                    <box>10,10,20,20</box>
                    <box>30,30,40,40</box>
                </bbox_list>
                更多描述
            </entity>
            引用<ref name="脏数据" id="1"/>
            </think><answer>D</answer>""",
            "expected": 0.6
        },
        {
            "desc": "陷阱: 空 bbox_list",
            "content": """<think>
            <entity name="空盒子" id="1"><bbox_list></bbox_list></entity>
            </think><answer>D</answer>""",
            "expected": 0.6 # 未达到0.7的要求 (至少一个box)
        },

        # --- Reference 复杂逻辑 ---
        {
            "desc": "复杂逻辑: 多个实体，其中一个未被引用",
            "content": """<think>
            <entity name="A" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>
            <entity name="B" id="2"><bbox_list><box>1,2,3,4</box></bbox_list></entity>
            这里引用了<ref name="A" id="1"/>，但是忘记引用B了。
            </think><answer>D</answer>""",
            "expected": 0.8
        },
        {
            "desc": "复杂逻辑: 引用先于定义 (时间悖论)",
            "content": """<think>
            我觉得是<ref name="A" id="1"/>。
            也就是<entity name="A" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>。
            </think><answer>D</answer>""",
            "expected": 0.8
        },
        {
            "desc": "复杂逻辑: 混合正确引用和错误引用 (严格模式)",
            "content": """<think>
            <entity name="A" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>
            正确引用: <ref name="A" id="1"/>
            错误引用: <ref name="A" id="1"/> (假设这里没问题)
            名字写错: <ref name="B" id="1"/>
            </think><answer>D</answer>""",
            "expected": 0.8 # 只要有一个ref不对，整体逻辑就判错
        },
        {
            "desc": "复杂逻辑: 多对多交替引用 (完美流)",
            "content": """<think>
            1. 发现<entity name="A" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>
            2. 发现<entity name="B" id="2"><bbox_list><box>1,2,3,4</box></bbox_list></entity>
            3. 比较<ref name="A" id="1"/>与<ref name="B" id="2"/>
            4. 确认<ref name="B" id="2"/>更大
            </think><answer>D</answer>""",
            "expected": 1.0
        },

        # --- 格式边界 ---
        {
            "desc": "边界: Entity紧贴think标签",
            "content": """<think><entity name="A" id="1"><bbox_list><box>0,1,2,3</box></bbox_list></entity> <ref name="A" id="1"/></think><answer>D</answer>""",
            "expected": 1.0
        },
        {
            "desc": "边界: 多个ref连在一起",
            "content": """<think>
            <entity name="A" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>
            <ref name="A" id="1"/><ref name="A" id="1"/>
            </think><answer>D</answer>""",
            "expected": 1.0
        },
        {
            "desc": "真实模拟: 完整的病理学推理片段",
            "content": """<think>
            观察切片，在中央可见一个<entity name="静脉" id="1"><bbox_list><box>100,100,200,200</box></bbox_list></entity>。
            其管壁较薄。
            旁边可见<entity name="胆管" id="2"><bbox_list><box>300,300,400,400</box></bbox_list></entity>，特征是单层立方上皮。
            对比<ref name="静脉" id="1"/>和<ref name="胆管" id="2"/>，
            可以看出<ref name="胆管" id="2"/>的细胞核更圆。
            所以选D。
            </think><answer>D</answer>""",
            "expected": 1.0
        }
    ]

    print(f"{'测试用例描述':<50} | {'预期':<5} | {'实际':<5} | {'状态'}")
    print("-" * 80)
    
    total = len(complex_cases)
    passed = 0
    
    for case in complex_cases:
        scores = scorer([case["content"]], solution, task, messages)
        actual = scores[0]
        
        # 允许微小的浮点误差
        is_pass = abs(actual - case["expected"]) < 1e-6
        status_str = f"{GREEN}PASS{RESET}" if is_pass else f"{RED}FAIL{RESET}"
        if is_pass: passed += 1
        
        print(f"{case['desc']:<50} | {case['expected']:<5} | {actual:<5} | {status_str}")
        
        if not is_pass:
            print(f"   {RED}>>> 调试提示: 此用例未通过，请检查正则匹配或逻辑顺序。{RESET}")

    print("-" * 80)
    print(f"测试完成: {passed}/{total} 通过")

run_tests()

测试用例描述                                             | 预期    | 实际    | 状态
--------------------------------------------------------------------------------
陷阱: ID重复 (1, 1, 2)                                 | 0.5   | 0.5   | PASS
陷阱: ID跳跃 (1, 3)                                    | 0.5   | 0.5   | PASS
鲁棒性: Entity内有换行和杂乱文本 (应判定<bbox_list>失败)            | 0.6   | 0.6   | PASS
陷阱: 空 bbox_list                                    | 0.6   | 0.6   | PASS
复杂逻辑: 多个实体，其中一个未被引用                                | 0.8   | 0.8   | PASS
复杂逻辑: 引用先于定义 (时间悖论)                                | 0.8   | 0.8   | PASS
复杂逻辑: 混合正确引用和错误引用 (严格模式)                           | 0.8   | 0.8   | PASS
复杂逻辑: 多对多交替引用 (完美流)                                | 1.0   | 1.0   | PASS
边界: Entity紧贴think标签                                | 1.0   | 1.0   | PASS
边界: 多个ref连在一起                                      | 1.0   | 1.0   | PASS
真实模拟: 完整的病理学推理片段                                   | 1.0   | 1.0   | PASS
---------------------------------

In [4]:
# ====== GPT测试样例构造 ======
msgs = [{"role": "user", "content": "dummy"}]

cases = [
    # 0.0：不满足基础格式（缺少<answer>）
    ("bad_basic", "<think>xxx</think> no answer", 0.0),

    # 0.5：满足基础格式，但没有任何<entity>
    ("basic_only_no_entity",
     "<think>xxx</think>yyy<answer>A</answer>", 0.5),

    # 0.6：有entity且id从1递增，但bbox不合规（缺bbox_list）
    ("id_ok_bbox_missing",
     '<think>发现<entity name="细胞核" id="1">no bbox</entity></think><answer>A</answer>', 0.6),

    # 0.6：多个entity但id不递增（跳号），仍应停在0.5（注意：此处有entity但id不合规）
    ("id_not_sequential",
     '<think>'
     '发现<entity name="细胞核" id="2"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.5),

    # 0.7：id递增 + bbox_list里至少一个box，但entity不都在think内（一个entity放到think外）
    ("bbox_ok_but_entity_outside_think",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '</think>'
     '并且还有<entity name="腺体" id="2"><bbox_list><box>5,6,7,8</box></bbox_list></entity>'
     '<answer>A</answer>', 0.7),

    # 0.8：id递增 + bbox ok + 所有entity都在think内，但ref缺失
    ("in_think_no_ref",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '以及<entity name="腺体" id="2"><bbox_list><box>5,6,7,8</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.8),

    # 0.8：ref在entity之前（违反“ref必须在对应entity后面”）
    ("ref_before_entity",
     '<think>'
     '<ref name="细胞核" id="1"/>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.8),

    # 0.8：ref在后面但name不一致
    ("ref_name_mismatch",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '</think>'
     '复述<ref name="细胞质" id="1"/>'
     '<answer>A</answer>', 0.8),

    # 1.0：满足全部要求（ref在对应entity之后，且name一致；每个entity至少一个ref）
    ("all_ok_score_1",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '以及<entity name="腺体" id="2"><bbox_list><box>5,6,7,8</box></bbox_list></entity>'
     '随后提到<ref name="细胞核" id="1"/>并再次提到<ref name="腺体" id="2"/>'
     '</think><answer>A</answer>', 1.0),
]

fmt = CGPOFormat()

completions = [c[1] for c in cases]
solutions = ["A"] * len(cases)
tasks = ["choice_rsn_loc"] * len(cases)
messages = [msgs] * len(cases)

rewards = fmt(completions, solutions, tasks, messages)

# 打印对照：样例名 | 期望 | 实际 | 是否通过
for (name, _content, expected), got in zip(cases, rewards):
    ok = (expected == got)
    print(f"{name:30s} expected={expected:.1f} got={got:.1f}  {'OK' if ok else 'FAIL'}")

bad_basic                      expected=0.0 got=0.0  OK
basic_only_no_entity           expected=0.5 got=0.5  OK
id_ok_bbox_missing             expected=0.6 got=0.6  OK
id_not_sequential              expected=0.5 got=0.5  OK
bbox_ok_but_entity_outside_think expected=0.7 got=0.7  OK
in_think_no_ref                expected=0.8 got=0.8  OK
ref_before_entity              expected=0.8 got=0.8  OK
ref_name_mismatch              expected=0.8 got=0.8  OK
all_ok_score_1                 expected=1.0 got=1.0  OK


In [5]:
# ====== GPT复杂测试样例构造 ======
msgs = [{"role": "user", "content": "dummy"}]

cases = [
    # 0.0：不满足基础格式（缺少<answer>）
    ("bad_basic", "<think>xxx</think> no answer", 0.0),

    # 0.5：满足基础格式，但没有任何<entity>
    ("basic_only_no_entity",
     "<think>xxx</think>yyy<answer>A</answer>", 0.5),

    # 0.6：有entity且id从1递增，但bbox不合规（缺<bbox_list>）
    ("id_ok_bbox_missing",
     '<think>发现<entity name="细胞核" id="1">no bbox</entity></think><answer>A</answer>', 0.6),

    # 0.5：存在entity但id不从1开始 / 不递增
    ("id_not_sequential",
     '<think>发现<entity name="细胞核" id="2"><bbox_list><box>1,2,3,4</box></bbox_list></entity></think><answer>A</answer>', 0.5),

    # 0.5：重复id（不是1..N严格递增）
    ("duplicate_id",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '又发现<entity name="腺体" id="1"><bbox_list><box>5,6,7,8</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.5),

    # 0.6：id递增OK，但bbox_list存在且为空（无<box>）
    ("bbox_list_empty_no_box",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),

    # 0.7：id递增 + bbox ok，但有entity在think外
    ("bbox_ok_but_entity_outside_think",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '</think>'
     '并且还有<entity name="腺体" id="2"><bbox_list><box>5,6,7,8</box></bbox_list></entity>'
     '<answer>A</answer>', 0.7),

    # 0.8：全部entity在think内，但完全没有ref
    ("in_think_no_ref",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '以及<entity name="腺体" id="2"><bbox_list><box>5,6,7,8</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.8),

    # 0.8：ref在entity之前（违反“ref必须在对应entity后面”）
    ("ref_before_entity",
     '<think>'
     '<ref name="细胞核" id="1"/>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.8),

    # 0.8：ref在entity后面但name不一致
    ("ref_name_mismatch",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '</think>'
     '复述<ref name="细胞质" id="1"/>'
     '<answer>A</answer>', 0.8),

    # 0.8：只有部分entity有ref（另一个缺ref）
    ("partial_ref_missing",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '以及<entity name="腺体" id="2"><bbox_list><box>5,6,7,8</box></bbox_list></entity>'
     '随后提到<ref name="细胞核" id="1"/>'
     '</think><answer>A</answer>', 0.8),

    # 1.0（复杂）：多box + 多次ref + 大量换行/空格，仍应通过
    ("all_ok_multi_box_multi_ref_with_whitespace",
     '<think>\n'
     '发现<entity name="脉管结构" id="1">\n'
     '  <bbox_list>\n'
     '    <box>286, 232, 788, 710</box>\n'
     '    <box>612, 673, 678, 913</box>\n'
     '  </bbox_list>\n'
     '</entity>\n'
     '接着发现<entity name="小叶间胆管" id="2">\n'
     '  <bbox_list>\n'
     '    <box>270, 30, 656, 441</box>\n'
     '    <box>668, 466, 869, 612</box>\n'
     '  </bbox_list>\n'
     '</entity>\n'
     '随后再次提到<ref name="脉管结构" id="1"/>，并多次复述<ref name="脉管结构" id="1"/>；\n'
     '也提到<ref name="小叶间胆管" id="2"/>。\n'
     '</think>\n'
     '<answer>D</answer>', 1.0),

    # 1.0（复杂）：ref不在think内（但在对应entity之后），应仍通过（你的实现允许）
    ("ref_after_think_before_answer",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '以及<entity name="腺体" id="2"><bbox_list><box>5,6,7,8</box></bbox_list></entity>'
     '</think>'
     '正文里再次提到<ref name="细胞核" id="1"/>与<ref name="腺体" id="2"/>'
     '<answer>A</answer>', 1.0),

    # 1.0（复杂）：ref放在<answer>之后/之内（只要在entity后面且name一致，你的实现会给1.0）
    ("ref_inside_answer_section",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '</think>'
     '<answer>A<ref name="细胞核" id="1"/></answer>', 1.0),

    # 1.0（复杂）：存在额外无关ref（id=99），应该判定为ref失败
    ("extra_irrelevant_ref_should_not_hurt",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '随后提到<ref name="细胞核" id="1"/>'
     '</think>'
     '插入无关<ref name="其他" id="99"/>'
     '<answer>A</answer>', 0.8),

    # 0.8（复杂）：ref出现在entity之后，但同id同时出现一个错误name（应失败）
    ("mixed_good_and_bad_ref_name_for_same_id",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '随后提到<ref name="细胞核" id="1"/>但又错误提到<ref name="细胞质" id="1"/>'
     '</think><answer>A</answer>', 0.8),
]

# ====== 仅新增/替换测试样例部分：覆盖“box坐标合法性”判定（由简单到复杂） ======
coord_cases = [
    # 0.6：最简单正确用例（单entity单box坐标合法），但不提供ref -> 最多到0.8？不对：ref缺失会卡在0.8
    # 这里我们专门让ref缺失，确保坐标合法时至少能过到0.8；但为最“简单”，也可只验证不被卡在0.6
    ("coord_ok_single_box_no_ref_should_reach_0_8",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1, 2, 3, 4</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.8),

    # 0.6：格式不匹配（缺逗号/少一项）
    ("coord_bad_format_missing_component",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1, 2, 3</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),

    # 0.6：包含非整数（小数）
    ("coord_bad_non_int_float",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1.5, 2, 3, 4</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),

    # 0.6：越界（-1 或 >1000）
    ("coord_bad_out_of_range_negative",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>-1, 2, 3, 4</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),
    ("coord_bad_out_of_range_gt_1000",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1, 2, 1001, 4</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),

    # 0.6：不满足严格不等（x1==x2 或 y1==y2）
    ("coord_bad_x1_eq_x2",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>10, 20, 10, 30</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),
    ("coord_bad_y1_eq_y2",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>10, 20, 30, 20</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),

    # 0.6：方向颠倒（x1>x2 或 y1>y2）
    ("coord_bad_x1_gt_x2",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>30, 20, 10, 40</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),
    ("coord_bad_y1_gt_y2",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>10, 40, 30, 20</box></bbox_list></entity>'
     '</think><answer>A</answer>', 0.6),

    # 0.7：多个box都合法 + 有ref，但ref包含未知id -> 根据你新增规则最终只能到0.8（先到0.8，无法到1.0）
    # 这里我们让所有条件到0.8，并验证不会被坐标卡在0.6
    ("coord_ok_multi_box_ref_unknown_id_should_0_8",
     '<think>'
     '发现<entity name="脉管结构" id="1"><bbox_list>'
     '<box>0,0,10,10</box><box>100,200,300,400</box>'
     '</bbox_list></entity>'
     '随后提到<ref name="脉管结构" id="1"/>'
     '</think>'
     '插入无关<ref name="其他" id="99"/>'
     '<answer>A</answer>', 0.8),

    # 0.6（复杂）：多个entity中只要有一个box不合法，就整体卡在0.6（后续不判）
    ("coord_mixed_entities_one_bad_should_0_6",
     '<think>'
     '发现<entity name="细胞核" id="1"><bbox_list><box>1,2,3,4</box></bbox_list></entity>'
     '以及<entity name="腺体" id="2"><bbox_list><box>10,20,10,30</box></bbox_list></entity>'  # x1==x2
     '随后提到<ref name="细胞核" id="1"/>并提到<ref name="腺体" id="2"/>'
     '</think><answer>A</answer>', 0.6),

    # 1.0（复杂）：多entity多box都合法 + ref齐全且无未知ref id
    ("coord_all_ok_multi_entity_multi_box_should_1_0",
     '<think>\n'
     '发现<entity name="细胞核" id="1"><bbox_list>\n'
     '<box>1, 2, 3, 4</box>\n'
     '<box>10, 20, 30, 40</box>\n'
     '</bbox_list></entity>\n'
     '发现<entity name="腺体" id="2"><bbox_list>\n'
     '<box>100, 200, 300, 400</box>\n'
     '</bbox_list></entity>\n'
     '随后提到<ref name="细胞核" id="1"/>与<ref name="腺体" id="2"/>'
     '</think><answer>A</answer>', 1.0),
]

# 将coord_cases拼到你原来的cases后面即可（或单独跑）
cases = cases + coord_cases

fmt = CGPOFormat()

completions = [c[1] for c in cases]
solutions = ["A"] * len(cases)
tasks = ["choice_rsn_loc"] * len(cases)
messages = [msgs] * len(cases)

rewards = fmt(completions, solutions, tasks, messages)

# 打印对照：样例名 | 期望 | 实际 | 是否通过
for (name, _content, expected), got in zip(cases, rewards):
    ok = (expected == got)
    print(f"{name:42s} expected={expected:.1f} got={got:.1f}  {'OK' if ok else 'FAIL'}")

bad_basic                                  expected=0.0 got=0.0  OK
basic_only_no_entity                       expected=0.5 got=0.5  OK
id_ok_bbox_missing                         expected=0.6 got=0.6  OK
id_not_sequential                          expected=0.5 got=0.5  OK
duplicate_id                               expected=0.5 got=0.5  OK
bbox_list_empty_no_box                     expected=0.6 got=0.6  OK
bbox_ok_but_entity_outside_think           expected=0.7 got=0.7  OK
in_think_no_ref                            expected=0.8 got=0.8  OK
ref_before_entity                          expected=0.8 got=0.8  OK
ref_name_mismatch                          expected=0.8 got=0.8  OK
partial_ref_missing                        expected=0.8 got=0.8  OK
all_ok_multi_box_multi_ref_with_whitespace expected=1.0 got=1.0  OK
ref_after_think_before_answer              expected=1.0 got=1.0  OK
ref_inside_answer_section                  expected=1.0 got=1.0  OK
extra_irrelevant_ref_should_not_hurt       expec

In [ ]:
import os
import os.path as osp
import json
import re
from tqdm import tqdm
from copy import deepcopy
import random

def load_json(path):
    with open(path, 'r') as jf:
        data = json.load(jf)
    return data

def dump_json(obj, path, indent=2):
    with open(path, 'w') as jf:
        json.dump(obj, jf, indent=indent, ensure_ascii=False)

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            data.append(json.loads(line))
    return data

print(len(load_json('/c22073/codes/ms-swift/zsxm_dataset/nips/3_rft/02_thumbnail_choice.json')))

In [ ]:
11294+7620

In [ ]:
# 调用LLM API的类
import os
import base64
import asyncio
from typing import List, Dict, Any, Union
from openai import OpenAI, AsyncOpenAI, APITimeoutError, APIConnectionError, RateLimitError
from tenacity import (
    retry, 
    stop_after_attempt, 
    wait_random_exponential, 
    retry_if_exception_type
)
from tqdm.asyncio import tqdm_asyncio

class MultiModalLLM:
    def __init__(self, provider="gemini", model=None, timeout=600.0, max_tokens=None):
        """
        :param provider: 'openai', 'gemini', 'deepseek'
        :param timeout: 全局超时时间（秒）
        """
        self.provider = provider.lower()
        self.timeout = timeout
        self.max_tokens = max_tokens
        
        # 配置各家厂商
        configs = {
            "openai": {
                "api_key": os.environ.get("OPENAI_API_KEY"),
                "base_url": None,
                "model": model or "gpt-5.2"
            },
            "gemini": {
                "api_key": os.environ.get("GEMINI_API_KEY"),
                "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
                "model": model or "gemini-3-flash-preview"
            },
            "deepseek": {
                "api_key": os.environ.get("DEEPSEEK_API_KEY"),
                "base_url": "https://api.deepseek.com",
                "model": model or "deepseek-chat"
            },
            "bailian": {
                "api_key": os.environ.get("BAILIAN_API_KEY"),
                "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
                "model": model or "qwen3-vl-plus"
            }
        }

        config = configs.get(self.provider)
        if not config or not config["api_key"]:
            raise ValueError(f"Provider {self.provider} 配置无效或缺少 API Key")

        self.model_name = config["model"]
        
        # 1. 同步客户端 (用于 chat_with_image)
        self.sync_client = OpenAI(
            api_key=config["api_key"],
            base_url=config["base_url"],
            timeout=self.timeout
        )
        
        # 2. 异步客户端 (用于 run_batch)
        self.async_client = AsyncOpenAI(
            api_key=config["api_key"],
            base_url=config["base_url"],
            timeout=self.timeout
        )

    def _encode_image(self, image_path):
        """读取图片并转Base64"""
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    def _prepare_payload(self, prompt, image_path, detail, json_mode):
        """内部工具：构造请求参数"""
        # 1. 处理 JSON 提示
        final_prompt = prompt
        if json_mode and "json" not in prompt.lower():
            final_prompt += " (Please output in JSON format)"

        # 2. 构造消息体
        messages = [{"role": "user", "content": [{"type": "text", "text": final_prompt}]}]
        if image_path:
            base64_image = self._encode_image(image_path)
            messages[0]["content"].append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}",
                    "detail": detail
                }
            })
        
        # 3. 构造参数字典
        kwargs = {
            "model": self.model_name,
            "messages": messages,
        }
        if self.max_tokens is not None:
            kwargs["max_tokens"] = self.max_tokens
        
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
            
        return kwargs

    @retry(
        retry=retry_if_exception_type((APITimeoutError, APIConnectionError)),
        wait=wait_random_exponential(min=1, max=10),
        stop=stop_after_attempt(3),
        reraise=True 
    )
    def chat_with_image(self, prompt: str, image_path: str = None, detail: str = "auto", json_mode: bool = False) -> str:
        """
        同步方法：直接调用，阻塞直到返回结果。
        无需 await，兼容旧代码。
        """
        kwargs = self._prepare_payload(prompt, image_path, detail, json_mode)
        
        # 使用 sync_client
        response = self.sync_client.chat.completions.create(**kwargs)
        return response.choices[0].message.content

    @retry(
        retry=retry_if_exception_type((APITimeoutError, APIConnectionError, RateLimitError)),
        wait=wait_random_exponential(min=1, max=90),
        stop=stop_after_attempt(5),
        reraise=True 
    )
    async def _chat_with_image_async(self, prompt: str, image_path: str = None, detail: str = "auto", json_mode: bool = False) -> str:
        """内部异步方法，用于并发"""
        kwargs = self._prepare_payload(prompt, image_path, detail, json_mode)
        
        # 使用 async_client
        response = await self.async_client.chat.completions.create(**kwargs)
        return response.choices[0].message.content

    async def run_batch(self, inputs: List[Dict[str, Any]], concurrency_limit: int = 10, json_mode: bool = False) -> List[Union[str, Dict]]:
        """
        异步批量处理。
        :param inputs: 参数列表
        :param concurrency_limit: 并发限制
        :param json_mode: 是否统一开启 JSON 模式 (如果不传，会优先看 inputs 里的 individual json_mode)
        """
        semaphore = asyncio.Semaphore(concurrency_limit)

        async def _safe_task(task_params):
            async with semaphore:
                try:
                    # 允许 inputs 中的字典单独覆盖 json_mode，否则使用全局传入的 json_mode
                    local_json_mode = task_params.get("json_mode", json_mode)
                    # 显式提取 key 避免传参报错
                    p = task_params.get("prompt")
                    img = task_params.get("image_path")
                    dtl = task_params.get("detail", "auto")
                    
                    return await self._chat_with_image_async(p, img, dtl, local_json_mode)
                except Exception as e:
                    return f"Error: {str(e)}"

        tasks = [_safe_task(params) for params in inputs]
        results = await tqdm_asyncio.gather(
            *tasks,
            total=len(tasks),
            desc="LLM batch progress"
        )
        return list(results)

# Test code
client = MultiModalLLM(provider="bailian", model='qwen3-vl-plus-2025-12-19')
print('-----------------')
print(client.chat_with_image('你好'))

In [ ]:
ilabel = [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 22043, 279, 21495, 17767, 70518, 57758, 448, 279, 25941, 17767, 34070, 43, 284, 856, 59, 701, 17767, 43, 65651, 284, 856, 488, 220, 23, 59, 701, 323, 17767, 42, 10994, 284, 856, 488, 220, 16, 15, 59, 701, 582, 1184, 311, 1477, 279, 897, 315, 17767, 87, 59, 3593, 1249, 1477, 17767, 87, 59, 701, 582, 1156, 6099, 429, 279, 2629, 315, 279, 25941, 304, 894, 21495, 374, 2677, 17767, 16, 23, 15, 24884, 43298, 59, 568, 15277, 11, 582, 646, 3270, 279, 23606, 1447, 78045, 93226, 488, 444, 42, 488, 46414, 284, 220, 16, 23, 15, 24884, 43298, 1124, 921, 78045, 856, 488, 320, 87, 488, 220, 16, 15, 8, 488, 320, 87, 488, 220, 23, 8, 284, 220, 16, 23, 15, 24884, 43298, 1124, 2533, 36192, 5740, 279, 3793, 389, 279, 2115, 3108, 11, 582, 633, 1447, 78045, 220, 18, 87, 488, 220, 16, 23, 284, 220, 16, 23, 15, 24884, 43298, 1124, 2533, 1249, 42123, 17767, 87, 59, 701, 582, 32256, 220, 16, 23, 504, 2176, 11067, 1447, 78045, 220, 18, 87, 284, 220, 16, 21, 17, 24884, 43298, 1124, 2533, 12209, 11, 582, 21749, 2176, 11067, 553, 220, 18, 1447, 78045, 856, 284, 220, 20, 19, 24884, 43298, 1124, 2533, 44500, 11, 279, 897, 315, 17767, 87, 57758, 374, 1124, 11520, 79075, 90, 20, 19, 11035, 568, 151645, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
nlabel = [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 22043, 279, 21495, 17767, 70518, 57758, 448, 279, 25941, 17767, 34070, 43, 284, 856, 59, 701, 17767, 43, 65651, 284, 856, 488, 220, 23, 59, 701, 323, 17767, 42, 10994, 284, 856, 488, 220, 16, 15, 59, 701, 582, 1184, 311, 1477, 279, 897, 315, 17767, 87, 59, 3593, 1249, 1477, 17767, 87, 59, 701, 582, 1156, 6099, 429, 279, 2629, 315, 279, 25941, 304, 894, 21495, 374, 2677, 17767, 16, 23, 15, 24884, 43298, 59, 568, 15277, 11, 582, 646, 3270, 279, 23606, 1447, 78045, 93226, 488, 444, 42, 488, 46414, 284, 220, 16, 23, 15, 24884, 43298, 1124, 921, 78045, 856, 488, 320, 87, 488, 220, 16, 15, 8, 488, 320, 87, 488, 220, 23, 8, 284, 220, 16, 23, 15, 24884, 43298, 1124, 2533, 36192, 5740, 279, 3793, 389, 279, 2115, 3108, 11, 582, 633, 1447, 78045, 220, 18, 87, 488, 220, 16, 23, 284, 220, 16, 23, 15, 24884, 43298, 1124, 2533, 1249, 42123, 17767, 87, 59, 701, 582, 32256, 220, 16, 23, 504, 2176, 11067, 1447, 78045, 220, 18, 87, 284, 220, 16, 21, 17, 24884, 43298, 1124, 2533, 12209, 11, 582, 21749, 2176, 11067, 553, 220, 18, 1447, 78045, 856, 284, 220, 20, 19, 24884, 43298, 1124, 2533, 44500, 11, 279, 897, 315, 17767, 87, 57758, 374, 1124, 11520, 79075, 90, 20, 19, 11035, 568, 151645, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
for i in range(1, max(len(ilabel), len(nlabel))):
    if ilabel[-i] != nlabel[-i]:
        print(i)
        break

In [ ]:
for i in range(1, len(ilabel)+1):
    if ilabel[-i] != -100:
        print(i)
        break
for i in range(1, len(nlabel)+1):
    if nlabel[-i] != -100:
        print(i)
        break

In [ ]:
79-53

In [ ]:
600-574

In [ ]:
import torch

# 定义变量，开启梯度
entropies_nan = torch.randn(5, requires_grad=True)
entropy_threshold = torch.tensor(0.5, requires_grad=True)

# 执行比较操作
entropy_mask = entropies_nan >= entropy_threshold

# 检查 mask 是否有关联的梯度函数
print(f"Mask values: {entropy_mask}")
print(f"Mask grad_fn: {entropy_mask.grad_fn}")  # 输出 None
print(f"Mask requires_grad: {entropy_mask.requires_grad}") # 输出 False

# 尝试反向传播会报错（或者因为没有 graph 而不执行任何更新）
# loss = entropy_mask.float().sum()
# loss.backward() # 会报错，因为 entropy_mask 没有 grad_fn

In [ ]:
import xml.etree.ElementTree as ET

# 模拟你的 XML 数据
xml_data = """<entity><name>细胞核</name><id>1</id><bbox_list><box>20, 0, 26, 16</box><box>20, 100, 34, 158</box></bbox_list></entity>
"""

def parse_entity(xml_string):
    # 解析字符串
    root = ET.fromstring(xml_string)
    print(dir(root))
    print(root.tail)
    
    # 1. 提取文本内容 ("细胞核")
    # .strip() 用于去除字符串前后的换行符和空格
    entity_name = root.text#.strip()
    
    # 2. 提取 <bbox> 标签内容
    bbox_element = root.find('id')
    print(bbox_element.text)
    bbox_text = bbox_element.text if bbox_element is not None else ""
    
    # 3. 将坐标字符串转换为数字列表 [1, 2, 3, 4]
    bbox_coords = [int(x) for x in bbox_text.split(',')] if bbox_text else []
    
    return entity_name, bbox_coords

# 执行解析
name, coords = parse_entity(xml_data)

# 打印结果
print(f"实体名称: {name}")
print(f"BBox 坐标: {coords}")